# **04 - Colombian loan portfolio: extraction and raw load from the datos.gov.co API**

## Goals
 - The main idea is to understand how we can identify customers in default (with Berka) and compare them with the Colombian loan portfolio, to give context about why customers could fall into default.   
 - Get the data from ‘www.datos.gov.co’ through an API (dataset `rvii-eis8`, Distribución de cartera por producto)
 - Load the data into the raw schema in Neon. Basically, we extract the information from the API and load it into Neon.
 - When the Superfinanciera publishes a new month, the project should fetch only that month, rather than everything all over again.

In [1]:
import pandas as pd
import requests
import sys
sys.path.append("..")
from src.load import get_engine, load_to_raw, verify_rows

Using the API URL we got from the web page, we send params written in SoQL, a language similar to SQL. The difference is the "$" sign at the beginning. We filter the column "fecha_corte" and limit the result to 5 rows.

Then, call requests.get and pass it the url and params.

In [2]:
url = 'https://www.datos.gov.co/resource/rvii-eis8.json'
params = {"$where": "fecha_corte='2026-07-31T00:00:00'", "$limit": 5}
response = requests.get(url, params=params)

In [3]:
print(response.status_code)   # code 200 means everything is ok
print(response.url)

200
https://www.datos.gov.co/resource/rvii-eis8.json?%24where=fecha_corte%3D%272026-07-31T00%3A00%3A00%27&%24limit=5


Turn the response into a DataFrame, passing it in JSON format with response.json().

In [4]:
df = pd.DataFrame(response.json())

In [5]:
df.shape

(5, 34)

In [6]:
df.dtypes    # All columns data are in str format

tipo_entidad                   str
codigo_entidad                 str
nombreentidad                  str
fecha_corte                    str
unicap                         str
descrip_uc                     str
renglon                        str
desc_renglon                   str
_1_saldo_de_la_cartera_a       str
_2_vigente                     str
_3_vencida_1_2_meses           str
_4_vencida_2_3_meses           str
_5_vencida_1_3_meses           str
_6_vencida_3_4_meses           str
_7_vencida_de_4_meses          str
_8_vencida_3_6_meses           str
_9_vencida_6_meses             str
_10_vencida_1_4_meses          str
_11_vencida_4_6_meses          str
_12_vencida_6_12_meses         str
_13_vencida_12_18_meses        str
_14_vencida_12_meses           str
_15_vencida_18_meses           str
_16_n_mero_de_clientes_mora    str
_17_calificaci_n_de_riesgo     str
_18_calificaci_n_de_riesgo     str
_19_calificaci_n_de_riesgo     str
_20_calificaci_n_de_riesgo     str
_21_calificaci_n_de_

We run a test with the following parameters:
- `$limit`: the maximum number of rows.
- `$offset`: where the rows start (0 is the first row). 
- `$order`: we pass `:id` because it is a unique identifier for each row. The API does not guarantee that rows come in the same order, so could be duplicated or lost. That is why we order by `:id`.

Then we pass the result to a DataFrame.

In [7]:
params = {"$limit": 3, "$offset": 3, "$order": ":id"}
response = requests.get(url, params=params)
df = pd.DataFrame(response.json())
df

,tipo_entidad,codigo_entidad,nombreentidad,fecha_corte,unicap,descrip_uc,renglon,desc_renglon,_1_saldo_de_la_cartera_a,_2_vigente,...,_17_calificaci_n_de_riesgo,_18_calificaci_n_de_riesgo,_19_calificaci_n_de_riesgo,_20_calificaci_n_de_riesgo,_21_calificaci_n_de_riesgo,_22_calificaci_n_de_riesgo,_23_calificaci_n_de_riesgo,_24_calificaci_n_de_riesgo,_25_calificaci_n_de_riesgo,_26_calificaci_n_de_riesgo
0,1,23,Banco De Occidente,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,108451564795,104345665139,...,9723,100426465880,324,2920771751,207,1936819277,165,1623502258,144,1544005629
1,1,57,Banco Pichincha S.A.,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,209307777743.65,196907738953.3,...,91873,196131426784.1,4802,6541005113.63,2385,2574770600.82,2515,3677977261.79,160,382597983.31
2,1,42,Davibank S.A. O Davibank.,2015-01-31T00:00:00.000,1,CRÉDITO ROTATIVO,5,CRÉDITO ROTATIVO TOTAL,629521014671.3,596115420556.47,...,64967,577605428295.04,2528,19860136454.86,1591,12667321103.07,2414,19128413216.22,45,259715602.11


The API gives us the data in batches of 1,000 rows, and the dataset has more than 100,000 rows, so we use a while loop. Each request returns one batch, which we save in `page`. If the page is empty, the loop breaks. If not, we add the rows to a list called `all_rows` with `.extend`. Then `offset` increases by 1,000 so the next request starts where the last one ended, and we print it to follow the progress.

In [8]:
page_size = 1000
offset = 0
all_rows = []
while True:
    params = {"$limit": page_size, "$offset": offset, "$order": ":id"}
    response = requests.get(url, params=params)
    page = response.json()
    if len(page) == 0:
        break
    all_rows.extend(page)
    offset += page_size
    print(offset)

1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000
76000
77000
78000
79000
80000
81000
82000
83000
84000
85000
86000
87000
88000
89000
90000
91000
92000
93000
94000
95000
96000
97000
98000
99000
100000
101000
102000
103000
104000
105000
106000
107000
108000
109000
110000
111000
112000


In [9]:
df = pd.DataFrame(all_rows) # it turns in a DataFrame
df.shape

(111787, 34)

To make sure no rows were lost or duplicated during pagination, we compare both totals. We use one parameter in params: `$select` with `count(*)`, so the API returns only the number of rows. We turn the result into an `int` because the API sends it as a `str`. Then we get the number of rows in `df` with `.shape[0]` and compare both numbers. If they do not match, it raises and error, se we don't continue with incomplete data.

In [10]:
params = {"$select": "count(*)"}
response = requests.get(url, params=params)
api_rows = int(response.json()[0]["count"])
df_rows = df.shape[0]
print(f"api_rows={api_rows}, df_rows={df_rows}, ok={api_rows == df_rows}")
if api_rows != df_rows:
    raise ValueError(f"api_rows:{api_rows} and df_rows:{df_rows} did not match")

api_rows=111787, df_rows=111787, ok=True


We reuse functions from src/load.py:
* `get_engine`: to connect to our database.
* `load_to_raw`: to load our data into our Neon database(`raw.co_loan_portfolio`), so we don’t need to call the API every time.
* `verify_rows`: to count the rows in Neon and compare them with the DataFrame. If they don't match, it raises and error.

In [11]:
engine = get_engine()
load_to_raw(df, "co_loan_portfolio", engine)
verify_rows(df, "co_loan_portfolio", engine)

co_loan_portfolio loaded
co_loan_portfolio, csv_rows=111787, db_rows=111787, ok=True


In [12]:
last_date = pd.read_sql("SELECT MAX(fecha_corte) FROM raw.co_loan_portfolio", engine).iloc[0, 0]

In [13]:
last_date

'2026-07-31T00:00:00.000'

We ask the API only for the rows with a cutoff date greater than `last_date`, with a limit of 50,000 so it doesn’t stop at 1,000 rows.

- With `requests.get` we pass the API URL and the params.
- Then we check the status code and the URL. If everything is ok, we save the result in `new_rows` with `response.json()` and check the number of rows. 0 means there is nothing new to load into the Neon database.
- If the result is not 0, it means the Superintendencia Financiera publishes a new month. After loading it, the result will be 0 again until the next month.

In [22]:
params = {"$where": f"fecha_corte>'{last_date}'", "$limit": 50000}
response = requests.get(url, params=params)
print(response.status_code)
print(response.url)
new_rows = response.json()
len(new_rows)

200
https://www.datos.gov.co/resource/rvii-eis8.json?%24where=fecha_corte%3E%272026-07-31T00%3A00%3A00.000%27&%24limit=50000


0

We test it with the previous month (June) to simulate a new month, as when the Superintendencia Financiera publishes new data. This is really important because it tests our incremental load.

- We use `test_` variables so we don’t change new_rows, which is the real one.
- Result: 818 rows, all from July. We don’t load them into Neon because July is already there.

In [23]:
test_date = '2026-06-30T00:00:00.000'
test_params = {"$where": f"fecha_corte>'{test_date}'", "$limit": 50000}
test_response = requests.get(url, params=test_params)
print(test_response.status_code)
print(test_response.url)
test_rows = test_response.json()
len(test_rows)

200
https://www.datos.gov.co/resource/rvii-eis8.json?%24where=fecha_corte%3E%272026-06-30T00%3A00%3A00.000%27&%24limit=50000


818

In [28]:
print(len(new_rows), len(test_rows))

0 818


In [27]:
test_rows

[{'tipo_entidad': '1',
  'codigo_entidad': '49',
  'nombreentidad': 'Av Villas',
  'fecha_corte': '2026-07-31T00:00:00.000',
  'unicap': '1',
  'descrip_uc': 'CRÉDITO ROTATIVO',
  'renglon': '5',
  'desc_renglon': 'CRÉDITO ROTATIVO TOTAL',
  '_1_saldo_de_la_cartera_a': '22145101372.71',
  '_2_vigente': '20266638529.6',
  '_3_vencida_1_2_meses': '242541606.47',
  '_4_vencida_2_3_meses': '69862264.58',
  '_5_vencida_1_3_meses': '0',
  '_6_vencida_3_4_meses': '0',
  '_7_vencida_de_4_meses': '0',
  '_8_vencida_3_6_meses': '487594443.41',
  '_9_vencida_6_meses': '1078464528.65',
  '_10_vencida_1_4_meses': '0',
  '_11_vencida_4_6_meses': '0',
  '_12_vencida_6_12_meses': '0',
  '_13_vencida_12_18_meses': '0',
  '_14_vencida_12_meses': '0',
  '_15_vencida_18_meses': '0',
  '_16_n_mero_de_clientes_mora': '152',
  '_17_calificaci_n_de_riesgo': '1945',
  '_18_calificaci_n_de_riesgo': '19724255525.28',
  '_19_calificaci_n_de_riesgo': '46',
  '_20_calificaci_n_de_riesgo': '546214146.19',
  '_21_cal